# CS 521 HW3,
Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install gdown
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt


Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchaudio-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-12.9.7-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 140.4 MB/s  0:00:0400:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 146.8 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [5]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [6]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [7]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)

def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    delta = torch.zeros_like(x)
    for I in range(steps):
      grad = input_grad(model, x, delta, y)
      delta = delta + (alpha*grad.sign())
      delta = delta.clamp(-eps,eps)
      delta = clip_box(x,delta)
    return x+delta


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    delta = torch.zeros_like(x)
    for I in range(steps):
      grad = input_grad(model, x, delta, y)
      delta = delta + (alpha*grad.sign()/l2_norm(grad))
      f = (eps / l2_norm(delta)).clamp(max=1.0)
      delta = delta*f
      delta = delta.clamp(-eps,eps)
      delta = clip_box(x,delta)
    return x+delta

In [8]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


def fgsm(model, x, y, eps,_steps,_eps):
    delta = torch.zeros_like(x)
    grad = input_grad(model, x, delta, y)
    grad = x + clip_box(x, eps * grad.sign())
    return grad.detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1 , "fgsm":fgsm}


def attack(model, x, y, norm, steps=10,eps=EPS):
    # The attack for one norm with the default eps and step size.
    alpha = eps["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * eps[norm] / steps
    return ATTACKS[norm](model, x, y, eps[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [9]:
def test_model(model, steps=50, restarts=2, batch_size=250,norms=NORMS,eps=EPS):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *norms, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in norms:
            robust = clean.clone()

            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps,eps)
                with torch.no_grad():
                    robust = robust &  (model(x_adv).argmax(1) == y)
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [10]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.3  l2 71.4  l1 22.8  union 22.8   (29 s)


{'clean': 78.7, 'linf': 48.3, 'l2': 71.4, 'l1': 22.8, 'union': 22.8}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [45]:
import numpy as np

def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, dynamic_beta=False, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    beta_ema = gp_beta if gp_beta is not None else 0.3
    beta_array = []
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                if dynamic_beta:
                    ce_nat = F.cross_entropy(model(x), y)
                    g_n = torch.autograd.grad(ce_nat, params)
                    g_a = torch.autograd.grad(loss, params)
                    with torch.no_grad():
                        beta = (ce_nat / (loss + ce_nat)).item()
                        beta_array.append(beta)
                        # beta_ema = 0.7 * beta_ema + 0.3 * beta  # init beta_ema = 0.3
                    for p, ga, gn in zip(params, g_a, g_n):
                        p.grad = beta * gradient_projection(gn, ga) + (1 - beta) * ga
                else:
                    g_a = torch.autograd.grad(loss, params)
                    g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                    for p, ga, gn in zip(params, g_a, g_n):
                        p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} mean_beta {np.mean(beta_array)} std_beta {np.std(beta_array)} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Q1

Normal and Linf loss

In [12]:
def normal_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


def linf_adv_loss(model, x, y, eps=8 / 255, steps=10):
    model.eval()
    x_adv = attack(model, x, y, "linf", steps)
    model.train()
    return F.cross_entropy(model(x_adv), y)

# Train

In [13]:
SEED=42
normal_model = adv_train(make_model(), normal_loss, epochs=5, seed=SEED)
pdg_trained_model = adv_train(make_model(), linf_adv_loss, epochs=5, seed=SEED)

epoch 1: loss 1.897 (2 s)
epoch 2: loss 1.559 (4 s)
epoch 3: loss 1.376 (6 s)
epoch 4: loss 1.228 (8 s)
epoch 5: loss 1.141 (10 s)
epoch 1: loss 2.196 (12 s)
epoch 2: loss 2.094 (24 s)
epoch 3: loss 2.050 (36 s)
epoch 4: loss 2.022 (48 s)
epoch 5: loss 2.011 (59 s)


# Eval

In [14]:
eps_array = [ (2**I)/255 for I in range(8)]

normal_eval_data = []
adv_eval_data=[]
norms = ["linf"]

for eps in eps_array:
    print('Normal Eval with eps = {}'.format(eps), end=' ')
    normal_eval_data.append(test_model(normal_model,norms=norms, eps={'linf':eps}))

    print('Adversarial Eval with eps = {}'.format(eps), end=' ')
    adv_eval_data.append(test_model(pdg_trained_model,norms=norms,eps={'linf':eps}))



Normal Eval with eps = 0.00392156862745098 clean 57.5  linf 36.8  union 36.8   (9 s)
Adversarial Eval with eps = 0.00392156862745098 clean 37.0  linf 34.5  union 34.5   (9 s)
Normal Eval with eps = 0.00784313725490196 clean 57.5  linf 21.2  union 21.2   (9 s)
Adversarial Eval with eps = 0.00784313725490196 clean 37.0  linf 32.8  union 32.8   (9 s)
Normal Eval with eps = 0.01568627450980392 clean 57.5  linf 5.0  union 5.0   (9 s)
Adversarial Eval with eps = 0.01568627450980392 clean 37.0  linf 30.2  union 30.2   (9 s)
Normal Eval with eps = 0.03137254901960784 clean 57.5  linf 0.3  union 0.3   (9 s)
Adversarial Eval with eps = 0.03137254901960784 clean 37.0  linf 23.5  union 23.5   (9 s)
Normal Eval with eps = 0.06274509803921569 clean 57.5  linf 0.0  union 0.0   (9 s)
Adversarial Eval with eps = 0.06274509803921569 clean 37.0  linf 11.9  union 11.9   (9 s)
Normal Eval with eps = 0.12549019607843137 clean 57.5  linf 0.0  union 0.0   (9 s)
Adversarial Eval with eps = 0.12549019607843137 

FGSM VS PGD

In [49]:
eps_array = [ (2**I)/255 for I in range(8)]

normal_eval_data = []
adv_eval_data=[]
norms = ["linf","fgsm"]

for eps in eps_array:
    print('Normal Eval with eps = {}'.format(eps), end=' ')
    normal_eval_data.append(test_model(normal_model,norms=norms, eps={'fgsm':eps}))

    print('Adversarial Eval with eps = {}'.format(eps), end=' ')
    adv_eval_data.append(test_model(pdg_trained_model,norms=norms,eps={'fgsm':eps}))



Normal Eval with eps = 0.00392156862745098 

KeyError: 'linf'

# Q3

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [16]:
import torch.nn.functional as F

def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    adv = attack(model,x,y,norm)
    model.train()
    return F.cross_entropy(model(adv), y)


In [17]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.212 (12 s)
epoch 2: loss 1.181 (24 s)
epoch 3: loss 1.155 (36 s)
epoch 4: loss 1.132 (48 s)
epoch 5: loss 1.104 (59 s)
epoch 6: loss 1.074 (71 s)
epoch 7: loss 1.051 (83 s)
epoch 8: loss 1.032 (95 s)
epoch 9: loss 1.016 (107 s)
epoch 10: loss 1.015 (119 s)
clean 78.5  linf 45.0  l2 70.4  l1 19.8  union 19.6   (29 s)


{'clean': 78.5, 'linf': 45.0, 'l2': 70.4, 'l1': 19.8, 'union': 19.6}

In [18]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.709 (12 s)
epoch 2: loss 0.630 (24 s)
epoch 3: loss 0.588 (36 s)
epoch 4: loss 0.546 (48 s)
epoch 5: loss 0.502 (60 s)
epoch 6: loss 0.457 (71 s)
epoch 7: loss 0.427 (83 s)
epoch 8: loss 0.393 (95 s)
epoch 9: loss 0.377 (107 s)
epoch 10: loss 0.372 (119 s)
clean 85.2  linf 30.2  l2 69.5  l1 9.9  union 9.7   (29 s)


{'clean': 85.2, 'linf': 30.2, 'l2': 69.5, 'l1': 9.9, 'union': 9.7}

In [19]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.229 (13 s)
epoch 2: loss 0.828 (27 s)
epoch 3: loss 0.591 (41 s)
epoch 4: loss 0.459 (54 s)
epoch 5: loss 0.361 (68 s)
epoch 6: loss 0.279 (81 s)
epoch 7: loss 0.227 (95 s)
epoch 8: loss 0.186 (108 s)
epoch 9: loss 0.166 (122 s)
epoch 10: loss 0.168 (136 s)
clean 84.2  linf 0.0  l2 1.0  l1 24.5  union 0.0   (29 s)


{'clean': 84.2, 'linf': 0.0, 'l2': 1.0, 'l1': 24.5, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [20]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    x_q = attack(model, x, y, "linf", steps=10)
    x_r = attack(model, x, y, "l1", steps=10)
    model.train()
    l_q = model(x_q)
    l_r = model(x_r)
    ce_q = F.cross_entropy(l_q, y, reduction="none")
    ce_r = F.cross_entropy(l_r, y, reduction="none")
    adv = torch.max(ce_q, ce_r).mean()
    with torch.no_grad():
        keep = l_q.argmax(1) == y
    if keep.any():
        log_p_q = F.log_softmax(l_q[keep], dim=1)
        log_p_r = F.log_softmax(l_r[keep], dim=1)
        kl = F.kl_div(log_p_r, log_p_q, reduction="batchmean", log_target=True)
    else:
        kl = torch.zeros((), device=y.device)
    return adv + lam * kl


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    f_n = g_n.flatten()
    f_a = g_a.flatten()
    denom = f_n.norm() * f_a.norm()
    if denom.item() <= 0:
        return torch.zeros_like(g_n)
    cos = (f_n @ f_a) / denom
    if cos.item() <= 0:
        return torch.zeros_like(g_n)
    return (cos * f_n).view_as(g_n)

In [47]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.34,dynamic_beta=False)
test_model(model_ramp)

/opt/conda/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.13/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/conda/lib/python3.13/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/conda/lib/python3.13/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


epoch 1: loss 1.425 mean_beta nan std_beta nan (27 s)
epoch 2: loss 1.328 mean_beta nan std_beta nan (54 s)
epoch 3: loss 1.295 mean_beta nan std_beta nan (80 s)
epoch 4: loss 1.264 mean_beta nan std_beta nan (107 s)
epoch 5: loss 1.234 mean_beta nan std_beta nan (134 s)
epoch 6: loss 1.206 mean_beta nan std_beta nan (161 s)
epoch 7: loss 1.183 mean_beta nan std_beta nan (188 s)
epoch 8: loss 1.169 mean_beta nan std_beta nan (215 s)
epoch 9: loss 1.154 mean_beta nan std_beta nan (242 s)
epoch 10: loss 1.147 mean_beta nan std_beta nan (268 s)
clean 78.5  linf 43.3  l2 68.1  l1 47.3  union 42.0   (29 s)


{'clean': 78.5, 'linf': 43.3, 'l2': 68.1, 'l1': 47.3, 'union': 42.0}

Pure Ramp - {'clean': 79.5, 'linf': 42.6, 'l2': 69.2, 'l1': 46.3, 'union': 40.6}
Ramp with 0.7 EMA Beta - {'clean': 78.5, 'linf': 43.3, 'l2': 68.4, 'l1': 47.2, 'union': 41.8}


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [22]:
def margins(logits, y):
    """z_y - max_{j != y} z_j, per example.  Negative means misclassified."""
    true_logit = logits.gather(1, y.view(-1, 1)).squeeze(1)
    other = logits.clone()
    other.scatter_(1, y.view(-1, 1), float("-inf"))
    return true_logit - other.max(1).values


def l2sp_penalty(model, base_params):
    total = 0.0
    for p, p0 in zip(model.parameters(), base_params):
        total = total + ((p - p0) ** 2).sum()
    return total


def my_loss(model, x, y, base_params=None, teacher=None, lam_sp=0.0, lam_kl=0.5,
            anchor_on=("clean", "linf"), topk=2, rho=0.5, select="margin",
            collect_stats=False):
    model.eval()
    x_adv = {n: attack(model, x, y, n, steps=10) for n in NORMS}
    model.train()

    logits = {n: model(x_adv[n]) for n in NORMS}
    ce = {n: F.cross_entropy(logits[n], y, reduction="none") for n in NORMS}
    ce_stack = torch.stack([ce[n] for n in NORMS], dim=1)              # (N, 3)
    margin_stack = torch.stack([margins(logits[n], y) for n in NORMS], dim=1)

    if select == "margin":
        # smaller margin = closer to flipping the label = more dangerous
        score = margin_stack
    elif select == "ce":
        # RAMP-style: larger cross-entropy = more dangerous
        score = -ce_stack
    else:
        raise ValueError(select)

    order = score.argsort(dim=1)          # most dangerous norm first
    loss = 0.0
    for r in range(topk):
        idx = order[:, r]
        picked = ce_stack.gather(1, idx.view(-1, 1)).squeeze(1)
        loss = loss + (1.0 if r == 0 else rho) * picked.mean()

    # --- functional anchor to the frozen base model ---------------------------------
    # The teacher is in eval() mode, so the student is also put in eval() mode for this
    # forward pass: otherwise the BatchNorm batch statistics of the student and the
    # running statistics of the teacher would be compared, which is an artefact and not
    # the anchor we want.  In eval() mode the anchor is 0 at initialisation (the student
    # *is* the base model) and grows with the drift, like a spring.
    if lam_kl > 0 and teacher is not None:
        teacher.eval()
        anchor_inputs = {"clean": x, "linf": x_adv["linf"], "l2": x_adv["l2"], "l1": x_adv["l1"]}
        was_training = model.training
        model.eval()
        for a in anchor_on:
            with torch.no_grad():
                log_p_teacher = F.log_softmax(teacher(anchor_inputs[a]), dim=1)
            log_p_student = F.log_softmax(model(anchor_inputs[a]), dim=1)
            loss = loss + lam_kl * F.kl_div(log_p_student, log_p_teacher, reduction="batchmean",
                                            log_target=True)
        if was_training:
            model.train()

    # --- weight-space anchor (L2-SP), the ablation of the functional anchor ---------
    if lam_sp > 0 and base_params is not None:
        loss = loss + lam_sp * l2sp_penalty(model, base_params)

    if collect_stats:
        _collect_stats(model, logits, ce, margin_stack, order, base_params)
    return loss


In [26]:
model_mine_0 = adv_train(load_base_model(), my_loss)
test_model(model_mine)

epoch 1: loss 2.030 (36 s)
epoch 2: loss 1.897 (73 s)
epoch 3: loss 1.843 (110 s)
epoch 4: loss 1.785 (147 s)
epoch 5: loss 1.721 (183 s)
epoch 6: loss 1.662 (220 s)
epoch 7: loss 1.609 (257 s)
epoch 8: loss 1.572 (293 s)
epoch 9: loss 1.540 (330 s)
epoch 10: loss 1.523 (367 s)
clean 77.8  linf 41.9  l2 68.5  l1 48.8  union 41.2   (29 s)


{'clean': 77.8, 'linf': 41.9, 'l2': 68.5, 'l1': 48.8, 'union': 41.2}

In [24]:
model_mine_1 = adv_train(load_base_model(), my_loss,seed=1)
model_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)


test_model(model_mine_1)
test_model(model_ramp_1)

epoch 1: loss 2.034 (36 s)
epoch 2: loss 1.896 (73 s)
epoch 3: loss 1.834 (109 s)
epoch 4: loss 1.777 (146 s)
epoch 5: loss 1.718 (183 s)
epoch 6: loss 1.664 (221 s)
epoch 7: loss 1.614 (258 s)
epoch 8: loss 1.575 (294 s)
epoch 9: loss 1.534 (331 s)
epoch 10: loss 1.528 (368 s)
epoch 1: loss 1.433 (27 s)
epoch 2: loss 1.342 (54 s)
epoch 3: loss 1.312 (81 s)
epoch 4: loss 1.283 (109 s)
epoch 5: loss 1.254 (137 s)
epoch 6: loss 1.225 (165 s)
epoch 7: loss 1.204 (193 s)
epoch 8: loss 1.193 (221 s)
epoch 9: loss 1.177 (248 s)
epoch 10: loss 1.169 (276 s)
clean 77.3  linf 42.3  l2 68.2  l1 48.2  union 41.1   (29 s)
clean 79.4  linf 42.4  l2 69.4  l1 46.5  union 40.4   (29 s)


{'clean': 79.4, 'linf': 42.4, 'l2': 69.4, 'l1': 46.5, 'union': 40.4}

In [48]:
model_ramp_0 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5,dynamic_beta=False,seed=0)
test_model(model_ramp_0)
model_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5,dynamic_beta=False,seed=1)
test_model(model_ramp_1)
model_b_ramp_0 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5,dynamic_beta=True,seed=0)
test_model(model_b_ramp_0)
model_b_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5,dynamic_beta=True,seed=1)
test_model(model_b_ramp_1)


epoch 1: loss 1.433 mean_beta nan std_beta nan (27 s)
epoch 2: loss 1.342 mean_beta nan std_beta nan (54 s)
epoch 3: loss 1.312 mean_beta nan std_beta nan (81 s)
epoch 4: loss 1.283 mean_beta nan std_beta nan (107 s)
epoch 5: loss 1.254 mean_beta nan std_beta nan (134 s)
epoch 6: loss 1.226 mean_beta nan std_beta nan (161 s)
epoch 7: loss 1.204 mean_beta nan std_beta nan (188 s)
epoch 8: loss 1.192 mean_beta nan std_beta nan (215 s)
epoch 9: loss 1.176 mean_beta nan std_beta nan (242 s)
epoch 10: loss 1.169 mean_beta nan std_beta nan (269 s)
clean 79.3  linf 42.7  l2 69.3  l1 46.3  union 40.8   (29 s)
epoch 1: loss 1.441 mean_beta nan std_beta nan (27 s)
epoch 2: loss 1.341 mean_beta nan std_beta nan (54 s)
epoch 3: loss 1.306 mean_beta nan std_beta nan (81 s)
epoch 4: loss 1.278 mean_beta nan std_beta nan (107 s)
epoch 5: loss 1.254 mean_beta nan std_beta nan (134 s)
epoch 6: loss 1.227 mean_beta nan std_beta nan (161 s)
epoch 7: loss 1.207 mean_beta nan std_beta nan (188 s)
epoch 8: 

{'clean': 78.8, 'linf': 42.8, 'l2': 68.7, 'l1': 46.1, 'union': 41.4}